#### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.arima.model import ARIMA
import warnings
import statsmodels.api as sm
from statsmodels.graphics.gofplots import qqplot
import numpy as np
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.stattools import adfuller, kpss
import pandas as pd
import warnings
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error


warnings.filterwarnings("ignore")

### Read Dataframe

In [18]:
denmark = pd.read_csv("data/structured_data/denmark_data1.csv")
df = denmark
df.rename(columns={"Real GDP growth": "Real GDP"}, inplace=True)
df = df.dropna()
df = df.set_index("Date")
df.index = pd.DatetimeIndex(df.index, freq=pd.infer_freq(df.index))
df["GDP growth"] = 100 * np.log(df["Real GDP"]).diff()
df.drop(columns=["Real GDP"], inplace=True)
MAX_LAGS = 4
TARGET   = 'GDP growth'

In [22]:
df.tail(8)

,Retail Sales,NET TRADE,Inflation (HICP/CPI),Unemployment rate,Interest rate 10 year government bond yield,Business Confidence Index,Stock Market Index,Consumer Confidence Index,Oil price,GFC,ip,GDP growth
Date,,,,,,,,,,,,
2022-04-01,-6.730452,-4.703200e+09,111.7,4.0,1.040,10.1,189.1463,-14.8,104.58,20930.04,110.0,1.792992
2022-07-01,-7.429519,2.358900e+09,114.9,4.6,1.586,5.9,182.3539,-18.1,111.93,21320.65,110.1,1.336999
2022-10-01,-8.732017,1.759800e+09,117.6,4.5,2.599,-1.2,167.3341,-23.5,93.33,21264.50,109.5,-1.347658
2023-01-01,-6.332993,1.064930e+10,116.4,4.4,2.433,-1.3,191.1422,-13.5,82.50,24357.97,120.1,1.173291
2023-04-01,-1.556420,6.797400e+09,117.6,5.0,2.597,-1.4,204.3422,-8.8,84.64,21918.30,119.4,-1.227452
2023-07-01,-0.573271,4.795000e+09,118.5,5.3,2.738,2.5,202.0943,-5.3,80.11,22093.78,119.3,-0.537373
2023-10-01,2.822581,6.307900e+09,117.7,6.2,3.133,1.8,207.3313,-6.7,90.60,22796.98,122.4,2.061204
2024-01-01,1.454017,5.579300e+09,117.8,6.0,2.403,3.0,226.6395,-4.7,80.12,21235.54,131.1,0.567894


### Main

In [3]:
corr = df.corr()
print(corr["GDP growth"].sort_values(ascending=False))

GDP growth                                     1.000000
Consumer Confidence Index                      0.228612
Retail Sales                                   0.170179
Business Confidence Index                      0.168178
Stock Market Index                             0.108799
Unemployment rate                              0.105313
NET TRADE                                      0.047519
Inflation (HICP/CPI)                           0.033070
ip                                             0.026571
Oil price                                      0.021583
GFC                                            0.009763
Interest rate 10 year government bond yield   -0.085579
Name: GDP growth, dtype: float64


In [4]:
data = df.copy()
X_columns = df.drop(columns=["GDP growth"]).columns
for col in X_columns:
    for lag in range(1, 5):  # 1 to 4 lags (quarters if quarterly data)
        data[f"{col}_lag{lag}"] = data[col].shift(lag)

In [6]:
corr = data.corr()
print(corr["GDP growth"].sort_values(ascending=False))

GDP growth                                          1.000000
Retail Sales_lag1                                   0.262584
Consumer Confidence Index                           0.228612
Retail Sales_lag2                                   0.213223
Retail Sales                                        0.170179
Business Confidence Index                           0.168178
Unemployment rate_lag3                              0.163993
Unemployment rate_lag2                              0.163374
Consumer Confidence Index_lag2                      0.123759
Retail Sales_lag4                                   0.113618
Stock Market Index                                  0.108799
Unemployment rate                                   0.105313
NET TRADE_lag3                                      0.097682
Retail Sales_lag3                                   0.092783
Unemployment rate_lag1                              0.082579
Consumer Confidence Index_lag3                      0.080940
Stock Market Index_lag2 

In [10]:
target_col = "GDP growth"

# Put here the candidate predictors you want the script to consider.
# I strongly recommend starting with a reduced list like this instead of every column.
candidate_exog = [
    "Retail Sales_lag1",
    "Retail Sales_lag2",
    "Consumer Confidence Index",
    "Business Confidence Index",
    "Unemployment rate_lag2",
    "Unemployment rate_lag3",
    "Interest rate 10 year government bond yield_lag2",
    "Interest rate 10 year government bond yield_lag3",
    "Oil price_lag2",
    "Oil price_lag3",
]

# ARIMA search space
p_values = range(0, 4)   # 0,1,2,3
d_values = [0]           # GDP growth already stationary
q_values = range(0, 4)   # 0,1,2,3

# Max number of exogenous variables in a model
max_exog_vars = 3

# Train/test split
test_size = 8   # change depending on your sample size and data frequency

# Optional: correlation filter among exogenous vars to reduce multicollinearity
max_corr_between_exog = 0.80

In [11]:
# =========================
# 2. PREPARE DATA
# =========================

# Keep only needed columns
used_cols = [target_col] + [c for c in candidate_exog if c in data.columns]
df = data[used_cols].copy()

# Drop rows with missing values
df = df.dropna().copy()

# Make sure everything is numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna().copy()

y = df[target_col]
X_all = df.drop(columns=[target_col])

if len(df) <= test_size + 10:
    raise ValueError(
        f"Not enough observations after dropping NA. "
        f"Need more than test_size + 10 rows, got {len(df)}."
    )

print(f"Observations used: {len(df)}")
print(f"Candidate exogenous variables available: {list(X_all.columns)}")

Observations used: 93
Candidate exogenous variables available: ['Retail Sales_lag1', 'Retail Sales_lag2', 'Consumer Confidence Index', 'Business Confidence Index', 'Unemployment rate_lag2', 'Unemployment rate_lag3', 'Interest rate 10 year government bond yield_lag2', 'Interest rate 10 year government bond yield_lag3', 'Oil price_lag2', 'Oil price_lag3']


In [12]:
# =========================
# 3. HELPER FUNCTIONS
# =========================

def is_valid_subset(X_subset, max_pairwise_corr=0.80):
    """
    Reject subsets with very high pairwise correlation.
    """
    if X_subset.shape[1] <= 1:
        return True
    
    corr_matrix = X_subset.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    max_corr = upper.max().max()
    
    if pd.isna(max_corr):
        return True
    return max_corr < max_pairwise_corr


def fit_arimax(y_train, X_train, order):
    """
    Fit SARIMAX as ARIMAX.
    """
    model = SARIMAX(
        y_train,
        exog=X_train,
        order=order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    result = model.fit(disp=False)
    return result


def evaluate_model(y_train, y_test, X_train, X_test, order):
    """
    Fit model and return AIC, BIC, and test RMSE.
    """
    result = fit_arimax(y_train, X_train, order)

    # Forecast on test set
    forecast = result.forecast(steps=len(y_test), exog=X_test)
    rmse = np.sqrt(mean_squared_error(y_test, forecast))

    return {
        "aic": result.aic,
        "bic": result.bic,
        "rmse": rmse,
        "result": result
    }

In [13]:
# =========================
# 4. TRAIN / TEST SPLIT
# =========================

train = df.iloc[:-test_size].copy()
test = df.iloc[-test_size:].copy()

y_train = train[target_col]
y_test = test[target_col]

X_train_all = train.drop(columns=[target_col])
X_test_all = test.drop(columns=[target_col])

In [14]:
# =========================
# 5. GENERATE EXOG SUBSETS
# =========================

all_exog_cols = list(X_train_all.columns)

exog_subsets = []
for k in range(1, max_exog_vars + 1):
    for combo in itertools.combinations(all_exog_cols, k):
        combo = list(combo)
        if is_valid_subset(X_train_all[combo], max_pairwise_corr=max_corr_between_exog):
            exog_subsets.append(combo)

print(f"Number of exogenous subsets to test: {len(exog_subsets)}")

Number of exogenous subsets to test: 139


In [15]:
# =========================
# 6. GRID SEARCH
# =========================

results_list = []

for exog_cols in exog_subsets:
    X_train = X_train_all[exog_cols]
    X_test = X_test_all[exog_cols]

    for order in itertools.product(p_values, d_values, q_values):
        try:
            metrics = evaluate_model(y_train, y_test, X_train, X_test, order)

            results_list.append({
                "order": order,
                "exog_vars": exog_cols,
                "n_exog": len(exog_cols),
                "aic": metrics["aic"],
                "bic": metrics["bic"],
                "rmse": metrics["rmse"]
            })

            print(
                f"Done | order={order} | exog={exog_cols} | "
                f"AIC={metrics['aic']:.2f} | RMSE={metrics['rmse']:.4f}"
            )

        except Exception:
            continue


if not results_list:
    raise ValueError("No models were successfully estimated.")


results_df = pd.DataFrame(results_list)

# Rank models: first by RMSE, then AIC
results_df = results_df.sort_values(by=["rmse", "aic"]).reset_index(drop=True)

print("\nTop 10 models:")
print(results_df.head(10))

Done | order=(0, 0, 0) | exog=['Retail Sales_lag1'] | AIC=300.39 | RMSE=1.6449
Done | order=(0, 0, 1) | exog=['Retail Sales_lag1'] | AIC=296.62 | RMSE=1.5083
Done | order=(0, 0, 2) | exog=['Retail Sales_lag1'] | AIC=294.88 | RMSE=1.6233
Done | order=(0, 0, 3) | exog=['Retail Sales_lag1'] | AIC=293.09 | RMSE=1.5938
Done | order=(1, 0, 0) | exog=['Retail Sales_lag1'] | AIC=298.83 | RMSE=1.5482
Done | order=(1, 0, 1) | exog=['Retail Sales_lag1'] | AIC=298.33 | RMSE=1.5494
Done | order=(1, 0, 2) | exog=['Retail Sales_lag1'] | AIC=295.75 | RMSE=1.6977
Done | order=(1, 0, 3) | exog=['Retail Sales_lag1'] | AIC=292.53 | RMSE=1.5959
Done | order=(2, 0, 0) | exog=['Retail Sales_lag1'] | AIC=298.33 | RMSE=1.5574
Done | order=(2, 0, 1) | exog=['Retail Sales_lag1'] | AIC=300.31 | RMSE=1.5672
Done | order=(2, 0, 2) | exog=['Retail Sales_lag1'] | AIC=293.84 | RMSE=1.5589
Done | order=(2, 0, 3) | exog=['Retail Sales_lag1'] | AIC=294.39 | RMSE=1.6137
Done | order=(3, 0, 0) | exog=['Retail Sales_lag1'] 

In [16]:
# =========================
# 7. FIT THE BEST MODEL AGAIN
# =========================

best_order = tuple(results_df.loc[0, "order"])
best_exog = results_df.loc[0, "exog_vars"]

print("\nBest model found:")
print("Order:", best_order)
print("Exogenous variables:", best_exog)

X_train_best = X_train_all[best_exog]
X_test_best = X_test_all[best_exog]

best_model = fit_arimax(y_train, X_train_best, best_order)

print("\nBest model summary:")
print(best_model.summary())


Best model found:
Order: (3, 0, 2)
Exogenous variables: ['Retail Sales_lag2', 'Unemployment rate_lag2', 'Interest rate 10 year government bond yield_lag3']

Best model summary:
                               SARIMAX Results                                
Dep. Variable:             GDP growth   No. Observations:                   85
Model:               SARIMAX(3, 0, 2)   Log Likelihood                -136.546
Date:                Wed, 25 Mar 2026   AIC                            291.091
Time:                        16:07:48   BIC                            312.752
Sample:                    01-01-2001   HQIC                           299.788
                         - 01-01-2022                                         
Covariance Type:                  opg                                         
                                                       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------

In [17]:
# =========================
# 8. FORECAST ON TEST SET
# =========================

best_forecast = best_model.forecast(steps=len(y_test), exog=X_test_best)

forecast_df = pd.DataFrame({
    "actual": y_test.values,
    "forecast": best_forecast.values
}, index=y_test.index)

print("\nForecast vs actual:")
print(forecast_df)

best_rmse = np.sqrt(mean_squared_error(y_test, best_forecast))
print(f"\nBest test RMSE: {best_rmse:.4f}")


# =========================
# 9. OPTIONAL: REFIT ON FULL SAMPLE
# =========================

X_full_best = X_all[best_exog]
final_model = SARIMAX(
    y,
    exog=X_full_best,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print("\nFinal model refit on full sample:")
print(final_model.summary())


Forecast vs actual:
              actual  forecast
Date                          
2022-04-01  1.792992  2.353202
2022-07-01  1.336999  0.777702
2022-10-01 -1.347658 -0.452725
2023-01-01  1.173291  0.017243
2023-04-01 -1.227452 -0.640198
2023-07-01 -0.537373 -0.335924
2023-10-01  2.061204  0.121383
2024-01-01  0.567894  0.230636

Best test RMSE: 0.9372

Final model refit on full sample:
                               SARIMAX Results                                
Dep. Variable:             GDP growth   No. Observations:                   93
Model:               SARIMAX(3, 0, 2)   Log Likelihood                -146.327
Date:                Wed, 25 Mar 2026   AIC                            310.653
Time:                        16:07:48   BIC                            333.152
Sample:                    01-01-2001   HQIC                           319.726
                         - 01-01-2024                                         
Covariance Type:                  opg                    

In [8]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =========================
# 1. SETTINGS
# =========================

target_col = "GDP growth"
best_exog = [
    "Retail Sales_lag2",
    "Unemployment rate_lag2",
    "Interest rate 10 year government bond yield_lag3"
]

candidate_orders = [
    (1, 0, 0),
    (2, 0, 0),
    (1, 0, 1),
    (0, 0, 0),   # useful benchmark: regression with exogenous vars only
]

test_size = 8


# =========================
# 2. PREPARE DATA
# =========================

df = data[[target_col] + best_exog].copy()

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna().copy()

train = df.iloc[:-test_size].copy()
test = df.iloc[-test_size:].copy()

y_train = train[target_col]
y_test = test[target_col]

X_train = train[best_exog]
X_test = test[best_exog]

y_full = df[target_col]
X_full = df[best_exog]


# =========================
# 3. FIT AND COMPARE MODELS
# =========================

results_summary = []
fitted_models = {}

for order in candidate_orders:
    try:
        model = SARIMAX(
            y_train,
            exog=X_train,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fitted = model.fit(disp=False)

        forecast = fitted.forecast(steps=len(y_test), exog=X_test)

        rmse = np.sqrt(mean_squared_error(y_test, forecast))
        mae = mean_absolute_error(y_test, forecast)

        results_summary.append({
            "order": order,
            "aic": fitted.aic,
            "bic": fitted.bic,
            "rmse": rmse,
            "mae": mae
        })

        fitted_models[order] = fitted

        print(f"\n==============================")
        print(f"Model: ARIMAX{order}")
        print(f"AIC:   {fitted.aic:.3f}")
        print(f"BIC:   {fitted.bic:.3f}")
        print(f"RMSE:  {rmse:.4f}")
        print(f"MAE:   {mae:.4f}")
        print("==============================")

    except Exception as e:
        print(f"Model {order} failed: {e}")


results_df = pd.DataFrame(results_summary).sort_values(by=["rmse", "aic"]).reset_index(drop=True)

print("\nModel comparison:")
print(results_df)


# =========================
# 4. CHOOSE BEST MODEL
# =========================

best_order = tuple(results_df.loc[0, "order"])
print(f"\nChosen model: ARIMAX{best_order}")

best_train_model = fitted_models[best_order]
print("\nBest model summary on training sample:")
print(best_train_model.summary())


# =========================
# 5. TEST FORECASTS
# =========================

best_forecast = best_train_model.forecast(steps=len(y_test), exog=X_test)

forecast_df = pd.DataFrame({
    "actual": y_test,
    "forecast": best_forecast
}, index=y_test.index)

print("\nForecast vs actual:")
print(forecast_df)

print("\nForecast errors:")
forecast_df["error"] = forecast_df["actual"] - forecast_df["forecast"]
print(forecast_df)


# =========================
# 6. REFIT CHOSEN MODEL ON FULL SAMPLE
# =========================

final_model = SARIMAX(
    y_full,
    exog=X_full,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(f"\nFinal ARIMAX{best_order} refit on full sample:")
print(final_model.summary())


# =========================
# 7. OPTIONAL: EXTRACT COEFFICIENTS
# =========================

coef_table = pd.DataFrame({
    "variable": final_model.params.index,
    "coef": final_model.params.values,
    "pvalue": final_model.pvalues.values
})

print("\nCoefficient table:")
print(coef_table)


Model: ARIMAX(1, 0, 0)
AIC:   295.255
BIC:   307.409
RMSE:  1.2682
MAE:   1.0557

Model: ARIMAX(2, 0, 0)
AIC:   292.852
BIC:   307.365
RMSE:  1.0228
MAE:   0.8064

Model: ARIMAX(1, 0, 1)
AIC:   291.758
BIC:   306.271
RMSE:  1.0370
MAE:   0.7692

Model: ARIMAX(0, 0, 0)
AIC:   302.340
BIC:   312.063
RMSE:  1.2085
MAE:   1.0083

Model comparison:
       order         aic         bic      rmse       mae
0  (2, 0, 0)  292.852054  307.365098  1.022822  0.806366
1  (1, 0, 1)  291.758099  306.271142  1.037042  0.769203
2  (0, 0, 0)  302.339829  312.063096  1.208476  1.008314
3  (1, 0, 0)  295.254537  307.408621  1.268181  1.055696

Chosen model: ARIMAX(2, 0, 0)

Best model summary on training sample:
                               SARIMAX Results                                
Dep. Variable:             GDP growth   No. Observations:                   85
Model:               SARIMAX(2, 0, 0)   Log Likelihood                -140.426
Date:                Wed, 25 Mar 2026   AIC                 

In [9]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =========================
# 1. SETTINGS
# =========================

target_col = "GDP growth"
exog_cols = [
    "Retail Sales_lag2",
    "Unemployment rate_lag2",
    "Interest rate 10 year government bond yield_lag3"
]

order = (1, 0, 0)   # Final chosen ARIMAX model
test_size = 8


# =========================
# 2. PREPARE DATA
# =========================

df = data[[target_col] + exog_cols].copy()

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna().copy()

train = df.iloc[:-test_size].copy()
test = df.iloc[-test_size:].copy()

y_train = train[target_col]
X_train = train[exog_cols]

y_test = test[target_col]
X_test = test[exog_cols]


# =========================
# 3. FIT MODEL ON TRAINING SAMPLE
# =========================

model = SARIMAX(
    y_train,
    exog=X_train,
    order=order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

results = model.fit(disp=False)

print("Training-sample model summary:")
print(results.summary())


# =========================
# 4. FORECAST ON TEST SAMPLE
# =========================

forecast = results.forecast(steps=len(y_test), exog=X_test)

forecast_df = pd.DataFrame({
    "actual": y_test,
    "forecast": forecast
}, index=y_test.index)

forecast_df["error"] = forecast_df["actual"] - forecast_df["forecast"]

rmse = np.sqrt(mean_squared_error(y_test, forecast))
mae = mean_absolute_error(y_test, forecast)

print("\nForecast vs actual:")
print(forecast_df)

print(f"\nTest RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")


# =========================
# 5. REFIT FINAL MODEL ON FULL SAMPLE
# =========================

y_full = df[target_col]
X_full = df[exog_cols]

final_model = SARIMAX(
    y_full,
    exog=X_full,
    order=order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print("\nFinal model refit on full sample:")
print(final_model.summary())


# =========================
# 6. COEFFICIENT TABLE
# =========================

coef_table = pd.DataFrame({
    "variable": final_model.params.index,
    "coef": final_model.params.values,
    "pvalue": final_model.pvalues.values
})

print("\nCoefficient table:")
print(coef_table)


# =========================
# 7. OPTIONAL: IN-SAMPLE FITTED VALUES
# =========================

df_results = df.copy()
df_results["fitted"] = final_model.fittedvalues

print("\nData with fitted values:")
print(df_results[[target_col, "fitted"]].tail(10))

Training-sample model summary:
                               SARIMAX Results                                
Dep. Variable:             GDP growth   No. Observations:                   85
Model:               SARIMAX(1, 0, 0)   Log Likelihood                -142.627
Date:                Wed, 25 Mar 2026   AIC                            295.255
Time:                        15:59:53   BIC                            307.409
Sample:                    01-01-2001   HQIC                           300.140
                         - 01-01-2022                                         
Covariance Type:                  opg                                         
                                                       coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------
Retail Sales_lag2                                    0.1329      0.055      2.395      0.017       0.02